# GeoEngine Dataset Export (system A)

Packs datasets that GeoEngine created from a processing graph (`datasetFromProcessingGraph`),
so that `geoengine_dataset_import.ipynb` can re-create them on another GeoEngine instance
(system B).

GeoEngine writes each such dataset into its own upload folder: the raster files (one per time
step and band) and a `dataset.json` holding the bodies of `POST /dataset` (`definition`) and
`POST /dataset/{name}/tiles` (`tiles`), with file paths relative to the folder. The export
copies these folders from A's storage and refreshes the **metadata** in `dataset.json` from
A's API, because tags, provenance and symbology are usually set after the dataset was written.

The package:

```
EXPORT_DIR/
  manifest.json            # source instance, files, sizes, SHA-256, spot checks
  <dataset name>/          # a copy of the dataset's upload folder
    dataset.json           # {"definition": {...}, "tiles": [...]}
    ...                    # the raster files, paths as listed in the tiles
```

In [ ]:
import datetime as dt
import hashlib
import json
import shutil
from pathlib import Path

import geoengine as ge
import requests

## Configuration

`A_UPLOAD_ROOT` is A's upload folder (`[upload] path` in its settings, relative to GeoEngine's
working directory unless absolute). It must be readable from where this notebook runs.

In [ ]:
GEOENGINE_URL = "http://localhost:3030/api"
GEOENGINE_USER = "admin@localhost"
GEOENGINE_PASSWORD = "adminadmin"

A_UPLOAD_ROOT = Path("/home/droenner/git/geoengine.worktrees/gdal-writer/geoengine/upload")

# Datasets to export: every dataset with this tag, plus any names listed explicitly.
SELECT_TAG = "nexgddp-yearly"
SELECT_NAMES = []

EXPORT_DIR = Path("out/export_nexgddp_yearly")
# Spot-check values from the aggregation notebook; entries for exported datasets are copied
# into the manifest so the import can re-check them on B. Missing file: no spot checks.
SPOTCHECKS_PATH = Path("out/nexgddp_yearly_spotchecks.json")

DATASET_FILE = "dataset.json"  # written by GeoEngine next to the raster files

In [ ]:
ge.initialize(GEOENGINE_URL, credentials=(GEOENGINE_USER, GEOENGINE_PASSWORD))
http = requests.Session()
http.headers.update(ge.get_session().auth_header)


def api(method, path, **kwargs):
    # Call the GeoEngine API and return the JSON body; raise with the server's message.
    response = http.request(method, f"{GEOENGINE_URL}/{path}", timeout=600, **kwargs)
    if not response.ok:
        raise RuntimeError(f"{method} {path}: {response.status_code} {response.text}")
    return response.json() if response.content else None


def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            digest.update(block)
    return digest.hexdigest()


source_info = api("GET", "info")
print(source_info)

## Select

The dataset listing is paged (at most 20 per page) and carries each dataset's tags. The tags are
filtered here, because the server's `tags` query parameter does not filter.

In [ ]:
PAGE = 20
names, offset = set(SELECT_NAMES), 0
while True:
    page = api("GET", "datasets", params={"order": "NameAsc", "offset": offset, "limit": PAGE})
    names.update(d["name"] for d in page if SELECT_TAG in (d.get("tags") or []))
    if len(page) < PAGE:
        break
    offset += PAGE
names = sorted(names)
print(f"{len(names)} datasets selected")

## Export

Per dataset: find its upload folder, copy it, and replace the properties in `dataset.json` with
the current ones from A's API. A dataset without upload folder or `dataset.json` (for example one
written by an older GeoEngine) stops the export. Re-running overwrites the package.

In [ ]:
PROPERTY_KEYS = ["name", "displayName", "description", "sourceOperator", "symbology", "provenance", "tags"]

EXPORT_DIR.mkdir(parents=True, exist_ok=True)
spotchecks = json.loads(SPOTCHECKS_PATH.read_text()) if SPOTCHECKS_PATH.exists() else []
manifest = {
    "exportedAt": dt.datetime.now(dt.timezone.utc).isoformat(),
    "source": {"url": GEOENGINE_URL, "info": source_info},
    "datasets": [],
    "spotchecks": [c for c in spotchecks if c["dataset"] in names],
}

for name in names:
    info = api("GET", f"dataset/{name}")
    data_path = info.get("dataPath")  # {"upload": id}, {"volume": name} or "external"
    upload = data_path.get("upload") if isinstance(data_path, dict) else None
    if upload is None:
        raise ValueError(f"{name}: not stored in an upload folder ({info.get('dataPath')})")
    source = A_UPLOAD_ROOT / upload
    if not (source / DATASET_FILE).is_file():
        raise FileNotFoundError(f"{name}: {source / DATASET_FILE} is missing")

    target = EXPORT_DIR / name
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(source, target)

    dataset = json.loads((target / DATASET_FILE).read_text())
    dataset["definition"]["properties"].update({k: info.get(k) for k in PROPERTY_KEYS})
    (target / DATASET_FILE).write_text(json.dumps(dataset, indent=1))

    files = sorted(p for p in target.rglob("*") if p.is_file() and p.name != DATASET_FILE)
    tile_paths = {t["params"]["filePath"] for t in dataset["tiles"]}
    missing = tile_paths - {p.relative_to(target).as_posix() for p in files}
    if missing:
        raise FileNotFoundError(f"{name}: tiles reference missing files {sorted(missing)[:3]}")

    manifest["datasets"].append({
        "name": name,
        "tiles": len(dataset["tiles"]),
        "files": [
            {"path": p.relative_to(EXPORT_DIR).as_posix(), "bytes": p.stat().st_size, "sha256": sha256(p)}
            for p in files
        ],
        "definitionSha256": sha256(target / DATASET_FILE),
    })
    print(f"exported {name}: {len(dataset['tiles'])} tiles, {sum(p.stat().st_size for p in files) / 1e6:.1f} MB")

(EXPORT_DIR / "manifest.json").write_text(json.dumps(manifest, indent=1))
print(f"manifest: {len(manifest['datasets'])} datasets, {len(manifest['spotchecks'])} spot checks")

## Check the package

Re-read the package and compare every file with its recorded checksum, so a broken package is
caught here and not on system B.

In [ ]:
total = 0
for entry in json.loads((EXPORT_DIR / "manifest.json").read_text())["datasets"]:
    assert sha256(EXPORT_DIR / entry["name"] / DATASET_FILE) == entry["definitionSha256"], entry["name"]
    for f in entry["files"]:
        assert sha256(EXPORT_DIR / f["path"]) == f["sha256"], f"{f['path']}: checksum mismatch"
        total += f["bytes"]
print(f"package OK: {len(names)} datasets, {total / 1e6:.1f} MB in {EXPORT_DIR.resolve()}")

## Transfer

Move the package into a GeoEngine volume on system B (a folder listed under `[data] volumes` in
B's settings), for example:

```
rsync -a --info=progress2 out/export_nexgddp_yearly/ B:/path/of/volume/nexgddp_yearly/
```

Then run `geoengine_dataset_import.ipynb` on B with `VOLUME_SUBDIR = "nexgddp_yearly"`.